In [0]:
%sql 
create catalog if not exists dbx;
create schema if not exists dbx.energy;

###Dim_Date

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbx.energy.dim_date (
    date_id     INT PRIMARY KEY,
    full_date   DATE NOT NULL
);


In [0]:
%sql
INSERT INTO dbx.energy.dim_date (date_id, full_date) VALUES
(1, '2024-01-10'),
(2, '2024-02-14'),
(3, '2024-03-22'),
(4, '2024-04-18'),
(5, '2024-05-30');


###Dim_Geography

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbx.energy.dim_geography (
    region_id   INT PRIMARY KEY,
    region_name VARCHAR(100) NOT NULL
);

In [0]:
%sql
INSERT INTO dbx.energy.dim_geography (region_id, region_name) VALUES
(1, 'Pacific Northwest'),
(2, 'Rocky Mountain'),
(3, 'Great Basin'),
(4, 'Southwest'),
(5, 'High Desert');

###Dim_Rate_Plan  


In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbx.energy.dim_rate_plan (
    rate_plan_id    INT PRIMARY KEY,
    plan_name       VARCHAR(100) NOT NULL,
    rate_type       VARCHAR(50)
);

In [0]:
%sql
INSERT INTO dbx.energy.dim_rate_plan (rate_plan_id, plan_name, rate_type) VALUES
(1, 'Residential Standard',  'Fixed'),
(2, 'Commercial Peak',       'Time-of-Use'),
(3, 'Industrial Bulk',       'Tiered'),
(4, 'Renewable Opt-In',      'Fixed'),
(5, 'Low-Income Assist',     'Subsidized');

###Dim_Location  

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbx.energy.dim_location (
    location_id INT PRIMARY KEY,
    city        VARCHAR(100),
    state       VARCHAR(50),
    zip_code    VARCHAR(20),
    region_id   INT REFERENCES dbx.energy.dim_geography(region_id)
);

In [0]:
%sql
INSERT INTO dbx.energy.dim_location (location_id, city, state, zip_code, region_id) VALUES
(1, 'Portland',    'OR', '97201', 1),
(2, 'Salt Lake City', 'UT', '84101', 2),
(3, 'Reno',        'NV', '89501', 3),
(4, 'Phoenix',     'AZ', '85001', 4),
(5, 'Boise',       'ID', '83701', 5);

###Dim_Customer  

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbx.energy.dim_customer (
    customer_id  INT PRIMARY KEY,
    full_name    VARCHAR(200) NOT NULL,
    rate_plan_id INT REFERENCES dbx.energy.dim_rate_plan(rate_plan_id),
    location_id  INT REFERENCES dbx.energy.dim_location(location_id),
    join_date    DATE
);


In [0]:
%sql
INSERT INTO dbx.energy.dim_customer (customer_id, full_name, rate_plan_id, location_id, join_date) VALUES
(1, 'Alice Morgan',    1, 1, '2020-03-15'),
(2, 'Bob Sanderson',   2, 2, '2019-07-22'),
(3, 'Carol Nguyen',    3, 3, '2021-01-10'),
(4, 'David Kim',       4, 4, '2018-11-05'),
(5, 'Emma Patel',      5, 5, '2022-06-30');

###Dim_Operator

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbx.energy.dim_operator (
    operator_id   INT PRIMARY KEY,
    operator_name VARCHAR(200) NOT NULL
);

In [0]:
%sql
INSERT INTO dbx.energy.dim_operator (operator_id, operator_name) VALUES
(1, 'PacifiCorp Operations West'),
(2, 'PacifiCorp Operations East'),
(3, 'Rocky Mountain Power Ops'),
(4, 'Pacific Power Ops'),
(5, 'Grid Solutions LLC');

###Dim_Power_Plant  

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbx.energy.dim_power_plant (
    plant_id    INT PRIMARY KEY,
    plant_name  VARCHAR(200) NOT NULL,
    fuel_type   VARCHAR(50),
    capacity_mw DECIMAL(10, 2),
    location_id INT REFERENCES dbx.energy.dim_location(location_id),
    operator_id INT REFERENCES dbx.energy.dim_operator(operator_id)
);


In [0]:
%sql
INSERT INTO dbx.energy.dim_power_plant (plant_id, plant_name, fuel_type, capacity_mw, location_id, operator_id) VALUES
(1, 'Colstrip Power Station',   'Coal',     2094.00, 1, 1),
(2, 'Hunter Power Plant',       'Coal',      1408.00, 2, 2),
(3, 'Currant Creek Power',      'Natural Gas', 495.00, 3, 3),
(4, 'Foote Creek Wind Farm',    'Wind',       135.00, 4, 4),
(5, 'Heber Geothermal Plant',   'Geothermal',  52.00, 5, 5);

###Fact_Energy_Consumption  

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbx.energy.fact_energy_consumption (
    consumption_id  INT PRIMARY KEY,
    date_id         INT REFERENCES dbx.energy.dim_date(date_id),
    customer_id     INT REFERENCES dbx.energy.dim_customer(customer_id),
    site_id         INT REFERENCES dbx.energy.dim_location(location_id),
    kwh_consumed    DECIMAL(12, 2)
);

In [0]:
%sql
INSERT INTO dbx.energy.fact_energy_consumption (consumption_id, date_id, customer_id, site_id, kwh_consumed) VALUES
(1, 1, 1, 1,  850.50),
(2, 2, 2, 2, 1230.75),
(3, 3, 3, 3,  540.00),
(4, 4, 4, 4, 3200.90),
(5, 5, 5, 5,  410.25);

###Fact_Energy_Item  

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbx.energy.fact_energy_item (
    energy_item_id  INT PRIMARY KEY,
    consumption_id  INT REFERENCES dbx.energy.fact_energy_consumption(consumption_id),
    date_id         INT REFERENCES dbx.energy.dim_date(date_id),
    customer_id     INT REFERENCES dbx.energy.dim_customer(customer_id)
);

In [0]:
%sql
INSERT INTO dbx.energy.fact_energy_item (energy_item_id, consumption_id, date_id, customer_id) VALUES
(1, 1, 1, 1),
(2, 2, 2, 2),
(3, 3, 3, 3),
(4, 4, 4, 4),
(5, 5, 5, 5);

###Fact_Energy_Generation  

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbx.energy.fact_energy_generation (
    generation_id       INT PRIMARY KEY,
    date_id             INT REFERENCES dbx.energy.dim_date(date_id),
    plant_id            INT REFERENCES dbx.energy.dim_power_plant(plant_id),
    downtime_minutes    INT
);

In [0]:
%sql
INSERT INTO dbx.energy.fact_energy_generation (generation_id, date_id, plant_id, downtime_minutes) VALUES
(1, 1, 1,   0),
(2, 2, 2,  30),
(3, 3, 3,  15),
(4, 4, 4,   0),
(5, 5, 5,  60);


###Fact_Grid_Reliability  

In [0]:
%sql
CREATE TABLE IF NOT EXISTS dbx.energy.fact_grid_reliability (
    incident_id         INT PRIMARY KEY,
    date_id             INT REFERENCES dbx.energy.dim_date(date_id),
    site_id             INT REFERENCES dbx.energy.dim_location(location_id),
    downtime_minutes    INT
);

In [0]:
%sql
INSERT INTO dbx.energy.fact_grid_reliability (incident_id, date_id, site_id, downtime_minutes) VALUES
(1, 1, 1,   0),
(2, 2, 2,  45),
(3, 3, 3,  20),
(4, 4, 4,   0),
(5, 5, 5,  90);